# Task 23 — Day 3 (30/10): Capstone Plans & Checklists, Exercises, Demo

**Author:** Elavarasu K &nbsp;|&nbsp; **Track:** ML / TensorFlow (Tasks 16–23) &nbsp;|&nbsp; **Date:** 30-10-2026

---

## Overview

This is the last slot of Task 23. No new training — it turns the results of the two earlier notebooks into the written deliverables:

| Part | What | Section |
|---|---|---|
| 1 | Results from the 28/10 and 29/10 runs (real MNIST, real IMDB, TF Serving in Docker, on my laptop CPU) | 1 |
| 2 | One-page **cost / latency / scale plan** for the sentiment API, **security** and **teardown** checklists | 2 |
| 3 | **"Which tool when"** guide and the **first-month monitoring** reflection | 3 |
| 4 | All **15 exercises** (12 and 13 computed in code) | 4 |
| 5 | **Demo** script and the final **deliverables checklist** | 5 |

Companion notebooks: `Task23_Day1_Phases1-8.ipynb` (Phases 1–8) and `Task23_Day2_Capstone.ipynb` (capstone build).

---
# 1. Results from the runs (my laptop, CPU only)

| Area | Result |
|---|---|
| Export | MNIST MLP 96.0 % test accuracy; reloaded SavedModel identical (max diff ~1e-8); raw-pixel model with `Rescaling` inside works; `py_function` model fails in a fresh process |
| REST vs gRPC | JSON = 1.9× binary for MNIST (mostly zero pixels), 5.1× for dense floats; 16 images of 224×224×3: REST 50.4 MB / 5.7 s, gRPC 9.6 MB / 0.1 s |
| Failure drills | all 8 bad requests (shape, dtype, signature, unknown model × REST/gRPC) turned into typed exceptions |
| Versioning | v2 hot-deployed with 0 errors over 120 polled requests; rollback by deleting `0002/` in 1.6 s |
| Canary (MNIST) | v1 96.6 % (535 req) vs v2 90.8 % (65 req), z = −2.28 → v2 significantly worse → roll back |
| Batching (MNIST, CPU) | 270 req/s off vs ~245–250 on → no gain for 1-image requests on a CPU |
| TFLite (MNIST) | 344 KB SavedModel → 313 KB float32 → 159 KB float16 → 83–85 KB int8, accuracy 95.97–96.00 % |
| Threads | 1 → 2 → 12 threads: 1.0× → 1.4× → 4.2× matmul throughput |
| Strategies | one device 3.04 s/epoch, Mirrored 4.18–4.40 s (0.69–0.73×), CentralStorage 3.78 s (0.80×) |
| LR scaling | 4 replicas: same lr 92.3 % → lr × 4 + warm-up 94.6 % |
| Multi-worker | 2 workers in lockstep, identical loss (1.4347 → 0.6958 → 0.5155) |
| Tuning | 10 random trials read from TensorBoard: best 3 layers, momentum 0.837, 92.4 % |
| Capstone (IMDB) | v1 84.0 %; REST/gRPC identical; Mirrored v2 1.01× (83.9 %); canary v2 81.6 % vs v1 84.0 % → rolled back; batching 274 → 280 req/s, p95 213 → 198 ms; TFLite 1255 → 630 → 317 KB, 84.0 / 84.0 / 83.8 % |

---
# 2. Capstone plans and checklists

### 2.1 Cost / latency / scale plan

**One-page plan — "Sentiment API v1"** (in the spirit of Task 15's production-readiness summary)

| Item | Plan |
|---|---|
| **What runs** | TF Serving container with the `sentiment` SavedModel (TextVectorization inside); REST for web/mobile clients, gRPC for internal batch callers; batching on with a 2 ms window |
| **Hardware** | **CPU**, not GPU: the model is ~0.3 M params and an embedding-average network — a GPU would sit idle and cost far more. 2 vCPU / 4 GB per replica |
| **Expected load** | design point 200 QPS steady, 1,000 QPS peak (assumption to validate) |
| **Replicas** | C5 measured **~280 req/s per laptop-CPU replica** (batching on). Plan at 60 % utilisation: 200 QPS ÷ (0.6 × 280) ≈ **2 replicas steady**, 1,000 QPS ÷ 168 ≈ **6 at peak**; minimum **2** (two zones) for availability; autoscale on CPU 60 % / p95 latency. Re-measure on the real server machine type before launch |
| **Latency budget** | p95 < 50 ms server-side, p99 < 150 ms; client timeout 1 s with one retry. My laptop load test was p95 ≈ 200 ms with 50 concurrent clients on one replica — that is why the plan scales out instead of loading one replica to 100 % |
| **Edge option** | the int8 TFLite core (317 KB, 83.8 % — measured) for an offline mobile feature, vocab shipped as an asset |
| **Training** | retrain monthly on fresh labelled data; the model trains in seconds per epoch on one CPU, so **one machine is plenty** — Mirrored gave 1.01× here, a cluster would only add cost and failure modes |
| **Monitoring** | QPS, p50/p95/p99, 4xx/5xx rate, replica count, CPU/mem; input drift (OOV-token rate, review length), prediction drift (share predicted positive); delayed-label accuracy weekly |
| **Rollback criteria** | auto-rollback if within the canary window: 5xx > 1 %, p95 > 1.5× baseline, predicted-positive share shifts > 10 pp, or canary accuracy significantly below v1 (z < −1.96) |

### 2.2 Security checklist

**Checklist:** no keys in code or repo (`.gitignore` covers `*.json` keys, `.env`) · service account with least privilege (predict-only) · TLS for every external hop (load balancer/ingress terminates TLS; mTLS internally if possible) · TF Serving ports **not** exposed to the internet, only to the gateway · buckets private, no secrets in names · rate limiting + request-size limit at the gateway (stops abuse and giant payloads) · input length capped (TextVectorization truncates at 200 tokens anyway) · dependency/image scanning, pinned image digest.

### 2.3 Teardown checklist

**Checklist:** `docker rm -f tf_serving_task23` (done by the last cell of the capstone notebook) · delete `task23_work/` if not needed · cloud (if used): undeploy + delete endpoints/models, delete buckets, delete service-account keys and accounts, stop/delete VMs and training jobs · check the billing report shows zero running resources.

### 2.4 Cost sketch (to fill with today's prices)

| Item | Assumption | How to price it |
|---|---|---|
| Serving nodes | 2 always-on small CPU nodes (min replicas), up to 6 at peak for ~2 h/day | node-hour price × (2 × 24 + 4 × 2) per day |
| Storage | ~5 MB model versions + logs | negligible; bucket price per GB-month |
| Network | 200 QPS × ~1 KB request/response | egress price per GB |
| Retraining | 1 CPU job per month, < 1 h | job-hour price |
| GPU | **not needed** | — |

Prices change by region and month, so they are looked up on the provider's pricing page on the day; a **budget alert** above the estimate catches mistakes.

---
# 3. Guide and reflection

### 3.1 "Which tool when"

| Tool | Latency | Throughput | Privacy | Cost | Ops burden | Use it when |
|---|---|---|---|---|---|---|
| Local `model.predict()` | lowest (in-process) | limited to one process | data stays local | ~0 | none | notebooks, batch scripts, prototypes |
| Flask/FastAPI service | medium (Python, JSON) | low–medium (GIL) | server-side | low | you own everything | non-TF logic (`py_function`, sklearn), quick internal demos |
| **TF Serving** (self-hosted) | low | high (C++, batching, gRPC) | server-side | VM/cluster cost | medium–high (k8s, monitoring) | production TF models, versioning/canaries, high QPS |
| Managed prediction service | low–medium (+cold start if scale-to-zero) | autoscaling | server-side, managed IAM | pay per node-hour | low | small teams, no infra people, spiky traffic |
| TFLite | very low, offline | per device | **on device** | free per inference | app release cycle | mobile/embedded/MCU, offline, battery-sensitive |
| TensorFlow.js | low, offline after load | per browser | **in browser** | free per inference | web deploy | interactive web features, privacy, no backend |
| Single GPU | — | good | — | low | low | **default for training** — try this first |
| Multi-GPU, one machine | — | very good (fast interconnect) | — | medium | low (`MirroredStrategy`) | model/data outgrow one GPU's speed |
| Multi-machine cluster | — | best at scale, but network-bound | — | high | high (TF_CONFIG, failures) | only when one machine with more/stronger GPUs isn't enough |
| TPU | — | excellent for large matmul-heavy models | — | medium–high | medium (XLA constraints) | big CNNs/Transformers, large batches |
| Cloud tuning service | — | parallel trials | — | per trial-hour | low | expensive models where smart (Bayesian) search saves many trials |

**Source's advice:** before building a cluster, try **a single machine with more or stronger GPUs** — less communication, fewer failure modes.

### 3.2 Reflection — what I would monitor in the first month

In month one, the biggest risks are not the model's test accuracy but the **gap between test data and live traffic** and **operational regressions**. I would watch three layers. **Service health:** QPS, p50/p95/p99 latency, error rates (4xx = bad clients/inputs, 5xx = us), container restarts and CPU/memory — a p95 regression or 5xx spike after a deploy triggers an **immediate rollback** (deleting the new version directory, which took 1.6 s in Phase 3). **Input drift:** distribution of review length, out-of-vocabulary token rate, language mix, compared with training statistics (Task 15); a rising OOV rate means the vocabulary is aging — a **retraining** trigger rather than a rollback. **Prediction drift and quality:** the share predicted positive and the score distribution (a sudden shift with stable inputs suggests a bug or skew; a gradual one suggests concept drift), and, as delayed labels arrive (user feedback, manual review samples), weekly accuracy versus the offline baseline. My own canaries showed why this matters: both MNIST v2 (90.8 % vs 96.6 %) and the sentiment v2 (81.6 % vs 84.0 %) looked fine offline but lost on live traffic, and only 10 % of users ever saw them. Rules: *rollback* when a new version is worse than the previous one on guardrails or canary metrics; *retrain* when the current version degrades over time against fresh labels or input drift crosses a threshold; *investigate* (not act) on single-day blips.

---
# 4. Exercises (Must Do)

**1. What does a SavedModel contain? How do you inspect it?** One or more **metagraphs** (graph + signatures), selected by **tags** (e.g. `serve`); `saved_model.pb` holds the graphs/signatures as a protobuf, `variables/` holds the weights (index + data shards), `assets/` holds extra files (vocabularies, class names), `assets.extra/` can hold warm-up requests. Inspect with `saved_model_cli show --dir … --all` (tags, signatures, input/output names, dtypes, shapes) and test with `saved_model_cli run` — Phase 1.6.

**2. When should you use TF Serving? Main features? Deployment tools?** When you need to serve TF models in production with high throughput and safe updates. Features: C++ high-performance server, REST + gRPC, multiple models and versions, automatic version discovery and hot-swap with quiescing, version pinning/policies, automatic batching, warm-up, monitoring endpoint. Deploy with Docker, Kubernetes (Deployments/Services/HPA, KServe), or a managed platform that runs it for you.

**3. How do you deploy a model across multiple TF Serving instances?** Run several identical TF Serving containers that read the same model store (shared bucket/volume), put a **load balancer** in front, and let an orchestrator (Kubernetes) handle health checks, rolling updates and autoscaling — or use a managed prediction service that does all of this (Phase 3.7).

**4. When gRPC rather than REST?** For large tensors and high throughput (binary protobuf, no JSON float bloat — I measured JSON at 1.9× the binary size for MNIST and 5.1× for dense floats, and 16 images of 224×224×3 took 5.7 s over REST vs 0.1 s over gRPC), for streaming/HTTP-2 multiplexing, and service-to-service calls. REST for simplicity, small payloads, browsers/`curl`, any-language clients.

**5. How does TFLite reduce model size?** FlatBuffer format; pruning training-only ops; constant folding and op fusion (e.g. BatchNorm folded); lower-precision weights — float16 (½), int8 weight quantization (¼); full-integer quantization of weights *and* activations (smaller and faster, integer-only hardware) — Phase 5.

**6. What is quantization-aware training and why?** Training with fake-quantization ops so the network learns to be robust to the rounding noise of int8 weights/activations; used when post-training quantization drops too much accuracy; ranges are learned, so calibration is automatic (Phase 5.5).

**7. Model vs data parallelism; why the latter?** Model parallelism splits one model across devices — architecture-specific, communication-heavy, often idle devices. Data parallelism replicates the model and splits the data — works for any model, little code, scales well until bandwidth saturates (Phase 7.1–7.2).

**8. Distribution strategies across multiple servers and how to choose?** `MultiWorkerMirroredStrategy` (synchronous, mirrored, AllReduce — default choice when each worker's GPUs and the network can hold full copies; good with fast interconnects), `ParameterServerStrategy` (asynchronous, central parameters — very large models/embeddings, heterogeneous or preemptible workers), `TPUStrategy` for TPU pods. Choose by model size (fits on each device?), network bandwidth, tolerance to stale gradients, and fault tolerance needs — and first ask whether one bigger machine would do.

**9. Train → deploy to TF Serving → query → v2 → roll back.** Done in Day 1 Phases 1–3 on MNIST: export `0001`, REST + gRPC clients, `0002` hot-deployed while a client polled every 100 ms (120 answers, **0 errors**), rollback by deleting `0002/` in **1.6 s**. Repeated end-to-end in the capstone (C1–C4) on the sentiment model, where the canary showed v2 worse and it was rolled back.

**10. MirroredStrategy vs CentralStorageStrategy.** Day 1 Phase 7.6, on 2 logical CPU devices: one device **3.04 s/epoch**, MirroredStrategy **4.18 s** (0.73×), CentralStorageStrategy **3.78 s** (0.80×). Both were slower than one device because the MLP is small (cheap steps, so coordination overhead dominates) and the two "devices" share the same physical cores. On real GPUs with a bigger model, Mirrored is expected to win. (No GPU available — done on logical CPU devices, as the exercise allows.)

**11. Hyperparameter tuning.** Done as the **local random-search analog** (no cloud billing account): 10 trials, 2 in parallel, metric read from TensorBoard event files; best **3 layers, momentum 0.837, 92.4 %**. The cloud tuning job is documented as a design (annotated YAML, Day 1 Phase 8.5).

In [1]:
import json
import numpy as np
import pandas as pd

def quantize_symmetric(w, bits=8):
    qmax = 2 ** (bits - 1) - 1                     # 127
    scale = np.abs(w).max() / qmax
    return np.round(w / scale).astype(np.int8), scale

# Exercise 12 - symmetric 8-bit quantization of [-1.5, -0.4, 0.0, 0.3, 0.8] by hand + check
w = np.array([-1.5, -0.4, 0.0, 0.3, 0.8])
m = np.abs(w).max(); scale = m / 127
by_hand = pd.DataFrame({"w": w, "w/m*127": (w / m * 127).round(3)})
by_hand["q = round()"] = np.round(w / m * 127).astype(int)
by_hand["dequantized = q*m/127"] = (by_hand["q = round()"] * scale).round(5)
by_hand["abs error"] = (by_hand["dequantized = q*m/127"] - w).abs().round(5)
q_np, s_np = quantize_symmetric(w)
assert (q_np == by_hand["q = round()"].values).all()
print(f"m = {m}, step = m/127 = {scale:.6f}; max abs error = {by_hand['abs error'].max():.5f} (<= step/2 = {scale/2:.5f})")
by_hand

m = 1.5, step = m/127 = 0.011811; max abs error = 0.00472 (<= step/2 = 0.00591)


,w,w/m*127,q = round(),dequantized = q*m/127,abs error
0,-1.5,-127.000,-127,-1.50000,0.00000
1,-0.4,-33.867,-34,-0.40157,0.00157
2,0.0,0.000,0,0.00000,0.00000
3,0.3,25.400,25,0.29528,0.00472
4,0.8,67.733,68,0.80315,0.00315


In [2]:
# Exercise 13 - JSON vs binary for 64 images of 224x224x3
n_floats = 64 * 224 * 224 * 3
binary_mb = n_floats * 4 / 1e6
assumed_chars = 18                          # e.g. "0.12345678901234567," ~ 17 digits + separator
est_json_mb = n_floats * assumed_chars / 1e6
X64 = np.random.default_rng(0).random((64, 224, 224, 3), dtype=np.float32)
measured_json_mb = len(json.dumps({"instances": X64.tolist()}).encode()) / 1e6
print(f"floats: {n_floats:,} | binary float32: {binary_mb:.1f} MB")
print(f"Estimate @ {assumed_chars} chars/float: {est_json_mb:.0f} MB (x{est_json_mb/binary_mb:.1f})")
print(f"Measured with real random pixels: {measured_json_mb:.0f} MB (x{measured_json_mb/binary_mb:.1f})")
X64_u8 = (X64 * 255).round()                # integers 0-255 sent as floats are much shorter
print(f"Same images as 0-255 integers in JSON: {len(json.dumps({'instances': X64_u8.astype(int).tolist()}).encode())/1e6:.0f} MB "
      "-> sending raw pixels (with Rescaling inside the model) also shrinks REST payloads")

floats: 9,633,792 | binary float32: 38.5 MB
Estimate @ 18 chars/float: 173 MB (x4.5)
Measured with real random pixels: 202 MB (x5.2)


Same images as 0-255 integers in JSON: 51 MB -> sending raw pixels (with Rescaling inside the model) also shrinks REST payloads


**14. Why do integer variables land on the CPU, and what does `set_soft_device_placement` change?** Placement needs a **kernel** for the op's dtype on the device; TF doesn't register GPU kernels for integer variables (and most int32 ops — int32 is used for shapes/indices that the host needs), so they are placed on `/cpu:0`. With soft placement **off**, explicitly asking for a device that can't run the op (or doesn't exist) raises an error; **on**, TF silently falls back to a device that can (Phase 6.4).

**15. 20 synchronous replicas, ignore the 2 slowest.** *Step time:* each step ends when the **18th** gradient arrives instead of the 20th → the long tail of stragglers (slow GPU, noisy neighbour, GC pause) no longer sets the pace, so step time becomes the 18th-order statistic — noticeably shorter and more stable. *Bandwidth:* all 20 replicas still compute, download parameters and usually send gradients, so traffic is barely reduced (slightly less aggregation work; the 2 late gradients are dropped). The cost is ~10 % of computed gradients wasted (an effective batch of 18/20). *Fault tolerance:* if a replica (or its machine) crashes, the aggregator still gets 18 gradients — training continues instead of hanging.

---
# 5. Demo

### 5.1 Two-minute spoken walkthrough (concept first, no slides)

1. **The problem.** "A model in a notebook helps no one. Task 23 is everything *around* the model: serving it, updating it safely, shrinking it for devices, and training it faster."
2. **Package it.** "I export a SavedModel into a numbered folder — the version *is* the folder, which makes deploy and rollback file operations. Preprocessing goes *inside* the model so clients can't forget it."
3. **Serve it.** "TF Serving in Docker answers over REST and gRPC. REST is simple; gRPC is binary — for large images it was 50× faster in my test because JSON turns every float into ~17 characters."
4. **Change it safely.** "I dropped v2 in while a client was polling: zero errors. Then a 10 % canary: v2 was significantly worse (z = −2.28), so I rolled back in under two seconds. Only 10 % of users ever saw it — that is the point of a canary."
5. **Shrink it.** "TFLite int8 made the model 4× smaller with no accuracy loss. Trap question: weight-only int8 is *smaller* but not *faster*, because weights are turned back into floats at run time; full-integer quantization is what runs in integer math."
6. **Train faster.** "Data parallelism copies the model to every device and averages gradients with AllReduce. On my CPU it was *slower* — small model, devices sharing cores — which shows the real lesson: communication cost decides whether more devices help."
7. **Scale out.** "I ran two real workers with TF_CONFIG; their losses were identical every epoch, proving synchronous training. Tuning services just read your metric from TensorBoard logs."
8. **Capstone + takeaway.** "The capstone did all of this on an IMDB sentiment model. My takeaway: keep it simple — one strong machine before a cluster, CPU when the model is small, and always canary before a full rollout."

### 5.2 Likely mentor questions — short answers

| Question | Answer |
|---|---|
| Why not just Flask? | TF Serving gives versioned hot-reload, batching, gRPC and C++ speed without me writing and owning that code |
| Why did batching not help? | 1-image requests on a CPU: the wait costs more than the grouping saves; on a GPU with a large model batching multiplies throughput |
| Why was Mirrored slower? | small model + logical devices on the same cores → overhead > compute saved |
| Why is int8 not faster? | weight-only quantization dequantizes at run time; activations must be quantized too (full-integer) for integer-speed inference |
| When would you use a cluster? | only when one machine with more/stronger GPUs is not enough — network bandwidth limits the speed-up |
| How do you roll back? | delete the new version folder (1.6 s in my test); in production, automated on guardrail metrics |

---
# Deliverables Checklist (whole Task 23)

| Deliverable | Where | Status |
|---|---|---|
| **Phase 1:** versioned SavedModel, tree, CLI show/run, bundled preprocessing, `py_function` failure, `export_model` + self-test | Day 1 | ✅ |
| **Phase 2:** TF Serving container, REST & gRPC clients, payload + latency benchmarks, failure drills + wrapper | Day 1 | ✅ live in Docker |
| **Phase 3:** v2 rollout + plot, rollback, canary A/B, batching charts, warm-up, scale-out diagram, runbook | Day 1 | ✅ live in Docker |
| **Phase 4:** managed service | Day 1 | ✅ **design document** (no billing account) |
| **Phase 5:** TFLite float32/16/dynamic/int8, NumPy quantization, TF.js page, target comparison | Day 1 | ✅ (QAT and the TF.js converter need extra packages not installed) |
| **Phase 6:** GPU verification, memory options, placement, execution walkthrough, thread pools, parallel models | Day 1 | ✅ (no GPU: memory options as scripts, multi-device on logical CPUs) |
| **Phase 7:** parallelism write-ups, sync/async, saturation estimate, strategy timings, custom loop, LR/batch experiment | Day 1 | ✅ |
| **Phase 8:** cluster spec/`TF_CONFIG`, 2-worker run, PS strategy, cloud job + tuning YAML, local tuning ranked | Day 1 | ✅ |
| **Capstone build:** export with preprocessing, REST+gRPC benchmark, Mirrored v2, canary + rollback, batching, TFLite | Day 2 | ✅ |
| **Capstone plans:** cost/latency/scale plan, security + teardown checklists | Day 3 §2 | ✅ |
| **"Which tool when" guide + monitoring reflection** | Day 3 §3 | ✅ |
| **Exercises 1–15** | Day 3 §4 | ✅ (11 done as local analog; cloud parts as design documents) |
| **Code** runs top to bottom, no secrets | all three | ✅ |